In [ ]:
experiment = "B"

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes

In [ ]:
!pip install fastapi uvicorn nest-asyncio pyngrok

In [ ]:
!pip uninstall -y bitsandbytes

In [ ]:
!pip install -U bitsandbytes

In [ ]:
!pip install transformers

In [ ]:
!ngrok config add-authtoken <API KEY COMES HERE>
!huggingface-cli login --token <API KEY COMES HERE>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#if the disk is full due to the huggingface cache memory
#!rm -rf /root/.cache/huggingface

In [ ]:
model_folder = f"./drive/MyDrive/Colab Notebooks/llama3_{experiment}/"

In [ ]:
from fastapi import FastAPI, Request

app = FastAPI()


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(model_folder)
model = AutoModelForCausalLM.from_pretrained(model_folder, torch_dtype=torch.float16).to("cuda")

In [ ]:
#base_model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", load_in_8bit=True)

In [ ]:
from pydantic import BaseModel

class InferenceInput(BaseModel):
    system: str
    user: str

In [ ]:
@app.post("/inference")
def predict(data: InferenceInput):
    output = infer(data.system, data.user)
    return {"response": output}

def infer(system_prompt, user_input):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

    # Apply chat template to format messages as a prompt
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1000,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            temperature=1.0,
            top_p=1.0,
            repetition_penalty=1.0
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # decoded = decoded.replace("</s>", "").strip()

    # # Debug info
    # print("Prompt: " + prompt + "#")
    # print("Raw: " + decoded)

    # Extract response from <|assistant|> onward
    if "assistant" in decoded:
        response_part = decoded.split("assistant")[-1].strip()
        # Optionally stop at next special token or new speaker tag

        # for stop_token in ["<|user|>", "<|system|>", "<|end|>", "<|"]:
        #     if stop_token in response_part:
        #         response_part = response_part.split(stop_token)[0].strip()
    else:
        response_part = decoded[len(prompt):].strip()

    return response_part



# system_prompt = "You are an expert in generating Java standard assertions. Your task is to insert assertions into a given method, ensuring the method’s correct behavior while keeping it compilable. Follow these instructions carefully:\nInput Method: A Java method will be provided, delimited by triple quotes (\"\"\"), where each line is numbered (starting from 1).\nTask: Insert Java standard assertions along with lines at which the assertion must be placed. Do not generate JUnit assertions. The remaining method lines after placing the inferred assertion will shift down by one to accommodate the assertion.\nExpectation: The purpose of these assertions must be to help developers comprehend the code by highlighting key assumptions, invariants, and expected program states. \nAssertions should be placed at locations where they improve readability and understanding of the method’s logic.\nThey should not be added arbitrarily or excessively—only where they clarify intended behavior.\nConstraints:\nGenerate only Java standard assertions. Avoid using any undefined methods, variable, or symbols in the project.\nThe assertions must use only variables, method or classes defined before the predicted line, ensuring the code remains compilable.\nDo not generate a new method definition., only focus on generating assertion and line number pairs.\nDo not generate assertions that require importing additional classes.\nAssertions must not alter the behavior of the method but validate the expected program state and program behavior at that program location.\nOutput Format:\nProvide output in pairs of assertions and line numbers.\nEach pair must be encapsulated within \u003cJAVA\u003e and \u003c/JAVA\u003e tags, formatted as (line_number, assertion). For instance: \u003cJAVA\u003e(3, assert a \u003c 3;)\u003c/JAVA\u003e, \u003cJAVA\u003e(5, assert a.getAge() \u003d\u003d 4;)\u003c/JAVA\u003e.\nExclude all descriptions, explanations, or any additional code (e.g., method structure or import statements). Only return the assertion and line number pairs."
# user_input = "The method for which you will generate assertions has the following characteristic(s):\n* Name: \"closeAllConnections\",\n* Signature: \"public void closeAllConnections()\"\n* Method declaration: \n\n\"\"\"\n1. public void closeAllConnections() {\n2.     AbstractHttpConnection connection \u003d first;\n3.     while (connection !\u003d null) {\n4.         AbstractHttpConnection next \u003d connection.next;\n5.         connection.close();\n6.         connection \u003d next;\n7.     }\n8. }\n\"\"\""
# user_input ="How are you?"
# print("Prompt:")
# print(user_input)

# print("\n---------------------------\nFine-Tuned Model:")
# print(infer(system_prompt, user_input))

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn

nest_asyncio.apply()

public_url = ngrok.connect(8000)
print("Public URL:", public_url)

uvicorn.run(app, port=8000)